# Somo la 11 - Itifaki ya Wakala kwa Wakala (A2A)


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Itifaki ya A2A ni nini?

**Itifaki ya Agent-to-Agent (A2A)** ni kiwango wazi kinachowawezesha maajenti wa AI kuwasiliana,
kugundua kila mmoja, na kushirikiana — hata kama wamejengwa kwenye mifumo tofauti au wamehifadhiwa
na huduma tofauti.

Dhana kuu:

- **Ugunduzi** – Maajenti huchapisha *Kadi ya Ajenti* inayobainisha uwezo wao, na kufanya iwe rahisi
  kwa maajenti wengine (au wasimamizi) kupata mtaalam sahihi kwa kazi fulani.
- **Upitishaji wa Ujumbe** – Maajenti hubadilishana jumbe zilizopangwa kupitia itifaki ya pamoja, kwa hivyo
  ombi kutoka kwa ajenti mmoja linaweza kueleweka na kutimizwa na ajenti mwingine bila kujali
  jinsi ilivyo ndani ya utekelezaji wake wa ndani.
- **Mzunguko wa Kazi** – A2A huainisha hali kama *imewasilishwa*, *inafanya kazi*, *imekamilika*, na
  *imeshindwa*, na kumpa msimamizi muonekano kamili jinsi kazi iliyoteuliwa inavyoendelea.

Katika somo hili tunajiruhusu kushirikiana kwa mtindo wa A2A kwa kuunganisha maajenti watatu wa usafiri
katika mchakato ambapo kila ajenti hutoa utaalamu wake na hupitisha matokeo kwa ajenti anayefuata.


## Kuunda Wakala Mahiri wa Kusafiri


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Ushirikiano wa Wakala Wingi Kupitia Mtiririko wa Kazi

Tunawaunganisha mawakala watatu katika mtiririko wa kazi mfululizo unaoiga upitishaji ujumbe wa A2A:

1. **CurrencyExchangeAgent** anapokea ombi la mtumiaji na kutoa mwongozo wa sarafu.
2. **ActivityPlannerAgent** anapokea muktadha ulioboreshwa na kuongezea mapendekezo ya shughuli.
3. **TravelManagerAgent** huunganisha pembejeo zote mbili katika muhtasari wa mwisho wa safari.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kuelewa A2A katika Uzalishaji

Katika mazingira ya uzalishaji itifaki ya A2A inafungua hali zenye nguvu za huduma kwa huduma:

| Uwezo | Maelezo |
|---|---|
| **Ushirikiano miongoni mwa mifumo mbalimbali** | Wakala aliyejengwa kwa mfumo mmoja anaweza kupeana majukumu kwa wakala aliyojengwa kwa mfumo mwingine wowote unaoendana na A2A, kuruhusu ushirikiano halisi miongoni mwa mashirika. |
| **Mipaka ya huduma** | Wakala wanaweza kuishi katika microservices tofauti, maeneo ya wingu tofauti, au hata mashirika tofauti huku wakishirikiana kwa urahisi. |
| **Ugunduzi wa nguvu** | Mratibu anaweza kuulizia rejista ya Kadi za Wakala wakati wa utekelezaji kutafuta mtaalamu aliye bora kwa kazi ndogo. |
| **Matiririko na arifa za push** | A2A inaunga mkono Matukio Yanayotumwa na Seva (SSE) kwa masasisho ya maendeleo kwa wakati halisi na arifa za push kwa kazi zinazochukua muda mrefu. |

Mchakato tuliojenga hapo juu ni toleo lililorahisishwa, ndani ya mchakato wa mfano huu. Katika uanzishaji halisi
kila wakala angeonyesha kiunganishi cha HTTP, kuchapisha Kadi ya Wakala, na kuwasiliana
kupitia itifaki ya A2A JSON-RPC.


## Muhtasari

Katika somo hili ulijifunza:

1. **Nini itifaki ya A2A** — kiwango cha wazi cha ugunduzi kati ya maajenti, ujumbe,
   na usimamizi wa kazi.
2. **Jinsi ya kuunda maajenti maalum** — maajenti wa Kubadilishana Sarafu, kupanga Shughuli,
   na msimamizi wa Usafiri.
3. **Jinsi ya kuunganisha maajenti ndani ya mtiririko wa kazi** — kwa kutumia `WorkflowBuilder` kuunda mfano wa kupitisha ujumbe mfululizo kati ya maajenti.

4. **Jinsi A2A inavyofanya kazi kwenye uzalishaji** — kuwezesha ushirikiano wa kuvuka mifumo, kuvuka huduma
   kwa ugunduzi wa mabadiliko na masasisho ya moja kwa moja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
